# 08 · Persistence and Time Travel

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama for §3 only; the rest runs offline.

A **checkpointer** saves the graph's state after every superstep. That one
feature gives you conversation memory, pause and resume (notebook 07), crash
recovery, and the ability to inspect, replay and **fork** any past moment of a
run. This notebook uses a real SQLite database and **real separate processes**
that get killed, so you can see what survives.

**You will learn**
- Threads, checkpoints and `StateSnapshot`: what exactly is saved
- `InMemorySaver` vs `SqliteSaver` vs Postgres
- Conversation memory that survives a process restart
- Crash recovery: **pending writes** and the three **durability modes**
- Time travel: browse history, **replay** from a checkpoint, **fork** with edited state
- Housekeeping: serialisation rules and deleting threads

**Prerequisites:** notebooks 02 and 07. *Optional background:* `00_CONCEPTS/02`
builds a checkpointer by hand, which makes this notebook feel familiar.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import sqlite3, subprocess, sys, textwrap, operator, pathlib
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.checkpoint.memory import InMemorySaver

WORK = pathlib.Path('checkpoints')          # this notebook's scratch folder (git-ignored)
WORK.mkdir(exist_ok=True)

## 1 · Why persistence is the foundation

Without a checkpointer, a graph run is a function call: when it returns or
crashes, everything is gone. With one:

| Capability | Needs from the checkpoint |
|---|---|
| Multi-turn conversations | the message history, per conversation |
| Human-in-the-loop (notebook 07) | the paused state and the question |
| Crash recovery | the last completed superstep, and which nodes are still due |
| Debugging and audit | every intermediate state |
| Time travel (replay, fork) | every checkpoint, addressable by id |

## 2 · The vocabulary

- **Thread**: one independent run or conversation, identified by `thread_id`
  (a user session, a ticket number, a job id). Checkpoints belong to a thread.
- **Checkpoint**: a snapshot taken after each superstep: the state values, which
  nodes run next, and metadata. Each has a `checkpoint_id`.
- **Pending writes**: if some nodes of a superstep finish and another fails, the
  finished nodes' outputs are saved too, so they don't have to run again.
- **`StateSnapshot`**: what `graph.get_state(config)` returns:

| Field | Meaning |
|---|---|
| `values` | the state at that checkpoint |
| `next` | the nodes due to run next (empty = finished) |
| `config` | contains `thread_id` and `checkpoint_id`, which you can pass back to address this exact point |
| `metadata` | `step` (superstep number), `source` (`input`, `loop`, `update`, …) |
| `tasks` / `interrupts` | pending work and any pending human questions |

**Which checkpointer?**

| Checkpointer | Package | Survives restart | Use for |
|---|---|---|---|
| `InMemorySaver` | built in | no | notebooks, tests |
| `SqliteSaver` | `langgraph-checkpoint-sqlite` | yes (a file) | single-machine apps, local tools |
| `PostgresSaver` | `langgraph-checkpoint-postgres` | yes (a server) | production, many workers |

All share one interface, so switching is a one-line change.

## 3 · Conversation memory that survives a restart

A chatbot compiled with a `SqliteSaver`. Each call sends **only the new
message**; the checkpointer supplies the history for that `thread_id`.

In [ ]:
from common import get_model
model = get_model()

DB = WORK / 'chat.db'
DB.unlink(missing_ok=True)

def chatbot(state: MessagesState) -> dict:
    return {'messages': [model.invoke([('system', 'Be brief.')] + state['messages'])]}

def build_chat(db_path):
    conn = sqlite3.connect(db_path, check_same_thread=False)   # graphs run nodes in threads
    b = StateGraph(MessagesState)
    b.add_node('chatbot', chatbot)
    b.add_edge(START, 'chatbot')
    return b.compile(checkpointer=SqliteSaver(conn))

chat = build_chat(DB)
ada = {'configurable': {'thread_id': 'user-ada'}}
bo = {'configurable': {'thread_id': 'user-bo'}}

print(chat.invoke({'messages': [('user', 'Hi, I am Ada and I build delivery robots.')]}, ada)['messages'][-1].content)
print(chat.invoke({'messages': [('user', 'What do I build?')]}, ada)['messages'][-1].content)
print(chat.invoke({'messages': [('user', 'What do I build?')]}, bo)['messages'][-1].content, '  ← another thread')

Now simulate a restart: throw away the graph object and the connection, build
everything again from the **same file**, and continue Ada's conversation:

In [ ]:
del chat
chat = build_chat(DB)                             # a "new process": nothing in memory
reply = chat.invoke({'messages': [('user', 'Remind me of my name.')]}, ada)
print(reply['messages'][-1].content)
print('messages stored for Ada:', len(reply['messages']))

## 4 · Inspecting a thread: state and history

`get_state` returns the latest snapshot; `get_state_history` returns every
checkpoint, newest first. A small deterministic graph makes the history easy
to read:

In [ ]:
class Calc(TypedDict):
    x: int

def add_one(s: Calc) -> dict:  return {'x': s['x'] + 1}
def double(s: Calc) -> dict:   return {'x': s['x'] * 2}
def square(s: Calc) -> dict:   return {'x': s['x'] ** 2}

b = StateGraph(Calc)
for fn in (add_one, double, square):
    b.add_node(fn.__name__, fn)
b.add_edge(START, 'add_one')
b.add_edge('add_one', 'double')
b.add_edge('double', 'square')
calc = b.compile(checkpointer=SqliteSaver(sqlite3.connect(WORK / 'calc.db', check_same_thread=False)))

cfg = {'configurable': {'thread_id': 'calc-1'}}
print('result:', calc.invoke({'x': 3}, cfg))
print()
print(f"{'checkpoint':<10} {'step':>4}  {'source':<6} {'next':<12} values")
for snap in calc.get_state_history(cfg):
    cid = snap.config['configurable']['checkpoint_id']
    print(f"{cid[-8:]:<10} {snap.metadata['step']:>4}  {snap.metadata['source']:<6} {str(snap.next):<12} {snap.values}")

Read it bottom-up: step −1 is the raw input, step 0 has `x=3` with `add_one`
next, and so on until step 3 with nothing next. **Every intermediate state is
kept**, which is what makes debugging ("what did the graph know before
`double` ran?") and time travel possible.

## 5 · Crash recovery, with real processes

The real test of durability is a process that **dies**. Below, the notebook
writes a small script, `crash_demo.py`, and runs it in separate Python
processes. The graph fans out to three fetches, then `write`. An environment
variable decides how it fails:

- `CRASH=fetch`: `fetch_c` raises an ordinary exception.
- `CRASH=write`: the process is killed instantly inside `write` (`os._exit`, like `kill -9`).

Every node appends its name to `runs.log`, so we can see exactly what re-ran.

In [ ]:
SCRIPT = WORK / 'crash_demo.py'
SCRIPT.write_text(textwrap.dedent("""
    import os, sys, sqlite3, operator
    from typing import TypedDict, Annotated
    from langgraph.graph import StateGraph, START, END
    from langgraph.checkpoint.sqlite import SqliteSaver

    class S(TypedDict):
        notes: Annotated[list[str], operator.add]
        report: str

    def log(name):
        with open('runs.log', 'a') as f:
            f.write(name + ' ')

    def fetcher(t):
        def fetch(s):
            log('fetch_' + t)
            if t == 'c' and os.environ.get('CRASH') == 'fetch':
                raise RuntimeError('fetch_c: upstream API error')
            return {'notes': [t]}
        return fetch

    def write(s):
        log('write')
        if os.environ.get('CRASH') == 'write':
            os._exit(1)                                   # the process dies instantly
        return {'report': '|'.join(sorted(s['notes']))}

    g = StateGraph(S)
    for t in 'abc':
        g.add_node('fetch_' + t, fetcher(t))
        g.add_edge(START, 'fetch_' + t)
        g.add_edge('fetch_' + t, 'write')
    g.add_node('write', write)
    g.add_edge('write', END)
    graph = g.compile(checkpointer=SqliteSaver(sqlite3.connect('crash.db', check_same_thread=False)))

    cfg = {'configurable': {'thread_id': 'job-1'}}
    try:
        if sys.argv[1] == 'start':
            out = graph.invoke({'notes': []}, cfg, durability=os.environ.get('DURABILITY', 'async'))
        else:
            out = graph.invoke(None, cfg)                 # None = continue from the last checkpoint
        print('   finished:', out)
    except Exception as err:
        print('   failed  :', err)
"""))


def run(mode, **env):
    import os
    result = subprocess.run([sys.executable, SCRIPT.name, mode], cwd=WORK, text=True,
                            capture_output=True, env={**os.environ, **env})
    print(f'  {mode:<6} exit={result.returncode} {result.stdout.strip()}')


def fresh():
    for f in ('crash.db', 'crash.db-wal', 'crash.db-shm', 'runs.log'):
        (WORK / f).unlink(missing_ok=True)

def ran():
    return (WORK / 'runs.log').read_text().strip()

**Case 1: a node raises.** `fetch_a` and `fetch_b` succeed in the same superstep
where `fetch_c` fails:

In [ ]:
fresh()
print('process 1 (fetch_c fails):')
run('start', CRASH='fetch')
print('process 2 (resume, API fixed):')
run('resume')
print('\nnodes executed across both processes:', ran())

`fetch_a` and `fetch_b` ran **once**. When `fetch_c` failed, their finished
results were saved as **pending writes**, so the resume ran only `fetch_c`, then
`write`. You don't lose (or pay again for) the work that succeeded.

**Case 2: the process is killed.** A kill can't be caught, so what survives
depends on *when* checkpoints reach the disk. That's the **durability** setting:

| `durability=` | When the checkpoint is written | Trade-off |
|---|---|---|
| `'async'` (default) | in the background while the next step starts | fast; a hard crash can lose the most recent steps |
| `'sync'` | before the next step starts | slower; nothing completed is lost |
| `'exit'` | only when the run finishes or pauses | fastest; a crash loses the whole run |

Let's kill the process inside `write` under each of the first two:

In [ ]:
for mode in ('async', 'sync'):
    fresh()
    print(f'durability={mode!r}')
    run('start', CRASH='write', DURABILITY=mode)
    run('resume')
    print('  nodes executed:', ran(), '\n')

With `'async'`, the kill landed before the background write of the fetch step
was on disk, so the resume **re-ran fetches that had already succeeded** (the
exact ones vary run to run). With `'sync'`, only `write` re-ran.

**How to choose:** if re-running a step is cheap and harmless, `async` is fine.
If steps are expensive (long LLM calls) or have side effects, use `sync`, and
make side-effecting nodes **idempotent** anyway, because the node that was
*running* when the process died always runs again. Durability means
*at-least-once*, never *exactly-once*.

## 6 · Time travel: replay and fork

Every checkpoint has an id, so you can go back to any point.

**Replay:** invoke with an old checkpoint's config and `None` as input. The
graph re-executes everything *after* that point. Use it to reproduce a bug from
a known state.

In [ ]:
history = list(calc.get_state_history(cfg))
before_double = next(s for s in history if s.next == ('double',))
print('going back to :', before_double.values, '→ next', before_double.next)
print('replay result :', calc.invoke(None, before_double.config))

**Fork:** change the state at an old checkpoint with `update_state`, then run
forward from the new branch. The original history is untouched. The fork is a
new checkpoint whose parent is the old one:

```
 input ── add_one ── (x=4) ─┬─ double ── square        original: x = 64
                            └─ [update x=10] ── double ── square   fork: x = 400
```

In [ ]:
forked_cfg = calc.update_state(before_double.config, {'x': 10})    # a new branch
print('fork result   :', calc.invoke(None, forked_cfg))
print('latest state  :', calc.get_state(cfg).values, ' ← the thread now continues from the fork')
print('checkpoints   :', len(list(calc.get_state_history(cfg))), 'in this thread (all branches kept)')

With an LLM in the graph, forking is how you answer "what if?": what if the
router had chosen differently, what if the retrieved document had been correct,
what if the human had approved. Edit that one value and rerun from there,
without repeating (or paying for) the steps before it.

`update_state` also takes `as_node='name'`, meaning "apply this update as if
that node had produced it". The graph then continues with that node's
successors. It's how an operator can manually complete a stuck step.

## 7 · Housekeeping

**What can be stored.** The default serialiser handles JSON types, messages,
Pydantic models, dataclasses, datetimes and more. It can't store live resources
(connections, clients, file handles, lambdas). Keep those out of state
(notebook 02 §8).

**Threads accumulate.** Every superstep of every thread adds a checkpoint. Delete
threads you no longer need:

In [ ]:
saver = calc.checkpointer
print('checkpoints before:', len(list(calc.get_state_history(cfg))))
saver.delete_thread('calc-1')
print('checkpoints after :', len(list(calc.get_state_history(cfg))))

**Memory across threads.** A checkpointer remembers *within* a thread. For facts
that should follow a user across conversations ("Ada prefers short answers"),
LangGraph has a separate **Store** (`compile(store=...)`) with namespaced
key-value and semantic search. Long-term memory is a big topic of its own, so
this section only notes that it exists.

## 8 · Trade-offs

- **Storage grows** with steps × threads × state size. Keep state lean, store
  large blobs elsewhere by reference, and delete finished threads.
- **Durability costs latency.** `sync` waits for a write on every step. It's
  negligible next to LLM calls, noticeable for tight loops of fast nodes.
- **Replaying runs the nodes again.** With LLMs the results can differ, and
  side-effecting nodes act again. Replay in a safe environment.
- **SQLite is single-machine.** Several worker processes writing the same file
  will contend; use Postgres for multi-worker deployments.

## 9 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| `InMemorySaver` in production | every conversation and paused run vanishes on restart | SQLite / Postgres |
| `sqlite3.connect(path)` without `check_same_thread=False` | errors when nodes run in threads | pass it (as above) |
| Reusing one thread id for unrelated runs | runs "remember" each other | one thread per conversation or job |
| Sending the whole history each turn **and** using a checkpointer | duplicated messages | send only the new message |
| Assuming a resume won't re-run anything | side effects repeat after a crash | idempotent nodes, `durability='sync'` |
| Never deleting threads | database grows forever | `delete_thread` or a retention policy |

## 10 · Check yourself

<details><summary><b>1.</b> What's the difference between a thread and a checkpoint?</summary>

A thread is one conversation or run (identified by `thread_id`). A checkpoint is one snapshot within a thread, taken after each superstep and identified by `checkpoint_id`. A thread has many checkpoints.
</details>

<details><summary><b>2.</b> In case 1, why didn't <code>fetch_a</code> and <code>fetch_b</code> run again?</summary>

They finished in the same superstep in which `fetch_c` failed, and their outputs were saved as pending writes. On resume only the failed node (and then `write`) needed to run.
</details>

<details><summary><b>3.</b> After a hard kill, the resume re-ran steps that had already finished. What setting explains that, and what would you change?</summary>

The default `durability='async'` writes checkpoints in the background, so a kill can lose the most recent ones. Use `durability='sync'` when re-running steps is expensive or unsafe.
</details>

<details><summary><b>4.</b> Even with <code>durability='sync'</code>, which node runs again after a crash, and what does that imply?</summary>

The node that was executing when the process died, because its result was never saved. Execution is at-least-once, so side-effecting nodes must be idempotent.
</details>

<details><summary><b>5.</b> Replay vs fork: what's the difference?</summary>

Replay continues from an old checkpoint as it was (re-executing later nodes). Fork first changes the state at that checkpoint with `update_state`, creating a new branch, and then runs from there. Both leave earlier history intact.
</details>

<details><summary><b>6.</b> A user says "the bot forgot everything after your deploy." What's the likely cause?</summary>

An in-memory checkpointer (or a database that isn't persisted between deploys). Conversation state must live in durable storage that outlives the process.
</details>

## Takeaway

A checkpointer saves state after every superstep, per **thread**. That gives
you memory across turns and restarts, pause and resume, recovery that keeps
finished work (**pending writes**), and **time travel**: inspect, replay and fork
any moment. Choose durability deliberately (`sync` when re-running is costly),
and remember recovery is at-least-once, so side effects must be idempotent.

Next → `09_streaming_and_debugging.ipynb`: watching graphs run in real time,
streaming tokens, and tracing.